# Demo 04: Single-Qubit Gate Dynamics

This notebook is a focused walkthrough for single-qubit gate simulation. It starts from one cosine-envelope `X90` pulse and checks the waveform, no-decoherence dynamics, state fidelity, leakage, Bloch trajectory, and a few optional extensions.

Conventions used in this demo:

- The drive path uses inductive coupling with `couple_type='induc'`.
- The default inductive operator model is `induc_phi_model='exact'`.
- Fidelity, leakage, and level populations are evaluated in the transmon eigenbasis `|g>, |e>, |f>, ...`, not in bare Fock indices.
- The main path does not load `T1` or `Tphi`; decoherence is compared in a separate section.


## 1. Setup

First we define imports, plotting defaults, and one reproducible `X90` working point. For the current code path, this point should reach high no-decoherence fidelity at the `99.99%` level with leakage around `1e-6` to `1e-5`.


In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt

from pysuqu.qubit import (
    SingleQubitGate,
    EnvelopeParams,
    PulseEvent,
    ChannelSchedule,
    MixerParams,
)

plt.rcParams.update({
    "figure.figsize": (8, 4),
    "axes.grid": True,
    "grid.alpha": 0.25,
})

TOTAL_TIME_NS = 100.0
SAMPLE_RATE_GSPS = 2.0
QUBIT_F01_GHZ = 5.0
QUBIT_ANHARM_GHZ = -0.25
ENERGY_TRUNC_LEVEL = 10

PULSE_START_NS = 20.0
PULSE_DURATION_NS = 40.0
LO_FREQ_GHZ = QUBIT_F01_GHZ
IF_FREQ_GHZ = QUBIT_F01_GHZ - LO_FREQ_GHZ

COUPLE_TYPE = "induc"
COUPLE_TERM = 0.8e-12
INDUC_PHI_MODEL = "exact"  # choose "exact" or "linear"

# Calibrated for the current no-decoherence, exact-sin(phi) path.
X90_AMP = 4.3423e-5
X90_DRAG = 0.08

# Keep the notebook fast by default. Set True for a small local amp/DRAG scan.
RUN_LOCAL_SCAN = False


## 2. Helper Functions

These helpers do two things:

1. Build the gate and the pulse channel.
2. Project simulation results onto the transmon eigenbasis so that population, leakage, and Bloch coordinates use the same computational subspace convention as `gate.calculate_fidelity(...)`.


In [ ]:
def build_x90_channel(
    *,
    amp: float = X90_AMP,
    drag: float = X90_DRAG,
    lo_freq: float = LO_FREQ_GHZ,
    if_freq: float = IF_FREQ_GHZ,
) -> ChannelSchedule:
    envelope = EnvelopeParams(
        name="x90_cosine_drag",
        duration=PULSE_DURATION_NS,
        peak_amp=amp,
        shape_type="cosine",
        drag_coeff=drag,
    )
    pulse = PulseEvent(
        name="X90",
        start_time=PULSE_START_NS,
        envelope=envelope,
        if_freq=if_freq,
        phase_offset=0.0,
    )
    return ChannelSchedule(
        name="q0_xy",
        sampling_rate=SAMPLE_RATE_GSPS,
        mixer_config=MixerParams(lo_freq=lo_freq),
        events=[pulse],
    )


def build_gate(channel: ChannelSchedule | None = None) -> SingleQubitGate:
    return SingleQubitGate(
        total_time=TOTAL_TIME_NS,
        sample_rate=SAMPLE_RATE_GSPS,
        qubit_frequency=QUBIT_F01_GHZ,
        qubit_anharmonicity=QUBIT_ANHARM_GHZ,
        nco_local=LO_FREQ_GHZ,
        energy_trunc_level=ENERGY_TRUNC_LEVEL,
        pulse_channel=channel,
    )


def as_density_matrix(state):
    return state * state.dag() if state.isket else state


def eigenstate_populations(gate: SingleQubitGate, result, levels: int = 4) -> np.ndarray:
    eigenstates = [gate.qubit.get_eigenstate(i) for i in range(levels)]
    rows = []
    for state in result.states:
        rho = as_density_matrix(state)
        rows.append([
            np.real(complex(eig.dag() * rho * eig))
            for eig in eigenstates
        ])
    return np.asarray(rows)


def qubit_bloch_trajectory(gate: SingleQubitGate, result) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    g_state = gate.qubit.get_eigenstate(0)
    e_state = gate.qubit.get_eigenstate(1)
    phase = np.exp(-1j * 2 * np.pi * gate.qubit.qubit_f01 * np.asarray(result.times))

    xyz = []
    for idx, state in enumerate(result.states):
        rho = as_density_matrix(state)
        pop_g = complex(g_state.dag() * rho * g_state)
        pop_e = complex(e_state.dag() * rho * e_state)
        coh_ge = complex(g_state.dag() * rho * e_state) * phase[idx]
        trace_ge = np.real(pop_g + pop_e)
        if trace_ge <= 0:
            xyz.append((np.nan, np.nan, np.nan))
            continue
        x = 2.0 * np.real(coh_ge) / trace_ge
        y = -2.0 * np.imag(coh_ge) / trace_ge
        z = np.real(pop_g - pop_e) / trace_ge
        xyz.append((x, y, z))
    return tuple(np.asarray(v) for v in zip(*xyz))


def print_metrics(label: str, metrics: dict) -> None:
    print(f"{label}")
    print(f"  fidelity       = {metrics['fidelity']:.9f}")
    print(f"  leakage        = {metrics['leakage']:.3e}")
    print(f"  phase error    = {metrics['phase_error_deg']:.4f} deg")


## 3. Build the X90 Pulse

This example uses a cosine envelope with DRAG. The simplest resonant setup is `LO_FREQ_GHZ = f01` and `IF_FREQ_GHZ = 0`. If you want to model a separate NCO and LO, keep `LO + IF = f01`.


In [ ]:
channel = build_x90_channel()
gate = build_gate(channel)

print(f"qubit f01      = {gate.qubit.qubit_f01:.6f} GHz")
print(f"anharmonicity  = {gate.qubit.qubit_anharm * 1e3:.3f} MHz")
print(f"LO + IF        = {LO_FREQ_GHZ + IF_FREQ_GHZ:.6f} GHz")
print(f"amp, drag      = {X90_AMP:.6g}, {X90_DRAG:.4f}")


In [ ]:
I_wave, Q_wave = gate.awg.generate_channel_waveform(channel)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(gate.awg.t_axis, I_wave, label="I")
ax.plot(gate.awg.t_axis, Q_wave, label="Q / DRAG")
ax.axvspan(PULSE_START_NS, PULSE_START_NS + PULSE_DURATION_NS, color="tab:blue", alpha=0.08)
ax.set_title("AWG IQ waveform for the X90 pulse")
ax.set_xlabel("Time (ns)")
ax.set_ylabel("Amplitude")
ax.legend()
plt.show()


## 4. No-Decoherence Simulation

We first run the gate with no `T1` or `Tphi`. The target state for this `X90` convention is

`|g> -> (|g> - i |e>) / sqrt(2)`

where `|g>` and `|e>` are eigenstates of the static transmon Hamiltonian.


In [ ]:
g_state = gate.qubit.get_eigenstate(0)
e_state = gate.qubit.get_eigenstate(1)
target_x90 = (g_state - 1j * e_state).unit()

result_no_decoh = gate.run_simulation(
    channel=channel,
    couple_term=COUPLE_TERM,
    couple_type=COUPLE_TYPE,
    induc_phi_model=INDUC_PHI_MODEL,
)

metrics_no_decoh = gate.calculate_fidelity(
    channel=channel,
    target_state=target_x90,
    result=result_no_decoh,
    couple_term=COUPLE_TERM,
    couple_type=COUPLE_TYPE,
    induc_phi_model=INDUC_PHI_MODEL,
    is_print=False,
)

print_metrics("No decoherence", metrics_no_decoh)


In [ ]:
pops = eigenstate_populations(gate, result_no_decoh, levels=4)
labels = ["g", "e", "f", "h"]

fig, ax = plt.subplots(figsize=(9, 4))
for idx, label in enumerate(labels):
    ax.plot(result_no_decoh.times, pops[:, idx], label=f"P_{label}")
ax.set_title("Eigenstate populations during the gate")
ax.set_xlabel("Time (ns)")
ax.set_ylabel("Population")
ax.set_ylim(-0.02, 1.02)
ax.legend(ncol=4)
plt.show()

print(f"final P_g + P_e = {pops[-1, 0] + pops[-1, 1]:.9f}")
print(f"final leakage   = {1.0 - (pops[-1, 0] + pops[-1, 1]):.3e}")


In [ ]:
x_bloch, y_bloch, z_bloch = qubit_bloch_trajectory(gate, result_no_decoh)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(x_bloch, y_bloch, lw=2)
ax.scatter([x_bloch[0], x_bloch[-1]], [y_bloch[0], y_bloch[-1]], c=["tab:green", "tab:red"], zorder=3)
ax.set_title("Rotating-frame Bloch projection (x-y plane)")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(-1.05, 1.05)
ax.set_ylim(-1.05, 1.05)
plt.show()

print(f"final Bloch vector = ({x_bloch[-1]:.4f}, {y_bloch[-1]:.4f}, {z_bloch[-1]:.4f})")


## 5. Add Decoherence as a Separate Check

This section reruns the same pulse after loading `T1` and `Tphi`. It is useful as a comparison, but the main question of gate calibration should be answered from the no-decoherence result above.


In [ ]:
gate.load_decoherence(
    T1=50e3,
    Tphi1=100e3,
    Tphi2=80e3,
)

result_with_decoh = gate.run_simulation(
    channel=channel,
    couple_term=COUPLE_TERM,
    couple_type=COUPLE_TYPE,
    induc_phi_model=INDUC_PHI_MODEL,
)

gate.clean_decoherence()

metrics_with_decoh = gate.calculate_fidelity(
    channel=channel,
    target_state=target_x90,
    result=result_with_decoh,
    couple_term=COUPLE_TERM,
    couple_type=COUPLE_TYPE,
    induc_phi_model=INDUC_PHI_MODEL,
    is_print=False,
)

print_metrics("No decoherence", metrics_no_decoh)
print_metrics("With T1/Tphi", metrics_with_decoh)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(
    ["no decoh", "with decoh"],
    [metrics_no_decoh["fidelity"], metrics_with_decoh["fidelity"]],
    color=["tab:green", "tab:orange"],
)
ax.set_title("Fidelity comparison")
ax.set_ylabel("Fidelity")
ax.set_ylim(0.999, 1.00005)
plt.show()


## 6. Optional: Local Amp/DRAG Scan

Enable this section by setting `RUN_LOCAL_SCAN = True` in the setup cell. It stays off by default so that the notebook remains quick to rerun from top to bottom.


In [ ]:
def evaluate_x90(amp: float, drag: float, *, model: str = INDUC_PHI_MODEL) -> dict:
    trial_channel = build_x90_channel(amp=amp, drag=drag)
    trial_result = gate.run_simulation(
        channel=trial_channel,
        couple_term=COUPLE_TERM,
        couple_type=COUPLE_TYPE,
        induc_phi_model=model,
    )
    metrics = gate.calculate_fidelity(
        channel=trial_channel,
        target_state=target_x90,
        result=trial_result,
        couple_term=COUPLE_TERM,
        couple_type=COUPLE_TYPE,
        induc_phi_model=model,
        is_print=False,
    )
    return {"amp": amp, "drag": drag, **metrics}


if RUN_LOCAL_SCAN:
    amp_values = X90_AMP * np.array([0.996, 1.000, 1.004])
    drag_values = X90_DRAG + np.array([-0.01, 0.0, 0.01])
    scan_rows = [evaluate_x90(float(amp), float(drag)) for amp in amp_values for drag in drag_values]
    best_row = max(scan_rows, key=lambda row: row["fidelity"])

    print("Local scan rows:")
    for row in scan_rows:
        print(
            f"amp={row['amp']:.7g}, drag={row['drag']:.4f}, "
            f"fid={row['fidelity']:.9f}, leak={row['leakage']:.3e}, "
            f"phase={row['phase_error_deg']:.4f} deg"
        )
    print()
    print("Best local point:")
    print_metrics(f"amp={best_row['amp']:.7g}, drag={best_row['drag']:.4f}", best_row)
else:
    print("Local scan skipped. Set RUN_LOCAL_SCAN = True in the setup cell to enable it.")


## 7. Exact vs Linear Inductive Drive Operator

`induc_phi_model='exact'` uses a matrix-function version of `sin(phi)` with the same expand-then-truncate pattern as the static Hamiltonian path. The `'linear'` option uses the small-angle approximation `phi`.

The comparison below keeps the same pulse parameters in both models. It therefore shows model sensitivity, not a fair re-optimization. If you switch models, you should usually recalibrate amplitude and DRAG.


In [ ]:
metrics_linear = evaluate_x90(X90_AMP, X90_DRAG, model="linear")

print_metrics("exact sin(phi)", metrics_no_decoh)
print_metrics("linear phi, same pulse parameters", metrics_linear)


## 8. Takeaways

- No-decoherence calibration should be judged in the transmon eigenbasis rather than in bare Fock indices.
- This demo is intentionally narrow: one `X90` pulse, one gate path, and one clean set of diagnostics.
- If you change `LO/IF`, `couple_term`, `induc_phi_model`, pulse length, or the transmission chain, it is normal to re-optimize amplitude and DRAG.
